# 55. 필수 근거 회수와 답변 반영의 분리 진단
승인된55–59계획의 첫 단계다. 기존52의40질문·280결과와54의96추출을 읽는다. 새 검색/임베딩/답변API는0이며 운영과 원본을 바꾸지 않는다. 기존52점수는 보존하고 이번 결과는 사후 개발 진단으로 표시한다.

각 필수 사실을 원문→JSON/raw DB/derived→QueryPlan→검색결과→실제LLM입력→답변에 연결한다. 카드별 실제 조건을 쓰고 같은 숫자의 다른 카드 근거를 대입하지 않는다. 직접 요구·중요 적용조건·부가정보·원문 불확실을 구분한다. 평가gold는 검색/추출/답변에 전달하지 않는다.

`55_fact_evaluation.py`는55–59에서 동일한 분모·0분모·반복macro·인용존재 검사를 재사용하기 위한 작은 오프라인 helper다. 원문 의미 판정과 실험 순서는 노트북이 소유한다. 인용 문자열의 존재 검사는 의미적 지원이나 판정자의 정확성을 자동 증명하지 않는다. 원문/카드 귀속은 판정자가 확인한다. HTML은 최종 결과를 확인한 뒤 직접 작성한다.

In [1]:
from pathlib import Path
import importlib.util,sys,json,time,collections,statistics,resource
_wall55=time.perf_counter();_cpu55=time.process_time()
ROOT=Path.cwd()
if not (ROOT/'notebooks').exists():ROOT=ROOT.parent
sys.dont_write_bytecode=True
spec=importlib.util.spec_from_file_location('fact_runtime55',ROOT/'notebooks/scripts/51_experiment_runtime.py')
X=importlib.util.module_from_spec(spec);sys.modules[spec.name]=X;spec.loader.exec_module(X)
X.load('recovery55','51_budget_recovery.py').install(X)
X.load('predicate55','52_query_predicate_correction.py').install(X)
E=X.load('fact_evaluation55','55_fact_evaluation.py');P=X.P
BASE=ROOT/'notebooks/data/52_corrected_four_retrieval_modes'
PATH=ROOT/'notebooks/data/55_fact_retrieval_generation_diagnosis';PATH.mkdir(exist_ok=True)
# Register only the newly approved experiment path; old write guards remain.
X.STAGES[55]=PATH.name;P.STAGES[55]=PATH.name
for common in [P.F,P.A.F,X.Q.P.F,X.Q.P.A.F]:common.STAGES[55]=PATH
P.F.initialize(55)
upstream=P.F.read_json(BASE/'execution_manifest.json');X.guard_manifest(upstream)
completed=P.F.read_json(BASE/'execution_results.json')
assert completed['manifest_sha256']==P.sha(upstream)
sealed=P.F.read_json(BASE/'evaluation_summary.json');seal_status=P.F.read_json(BASE/'completion_status.json')
E.validate_seal(sealed,seal_status,execution_sha=P.sha(completed),judgments_sha=P.sha(P.F.read_json(BASE/'question_source_judgments.json')),summary_sha=P.sha(sealed))
rows=completed['answers'];retrieval_rows=completed['retrievals']
judgment_rows=P.F.read_json(BASE/'question_source_judgments.json')['records']
retrievals={r['identity']:r for r in retrieval_rows}
old_judgments={r['identity']:r for r in judgment_rows}
queries=P.F.read_json(BASE/'questions.json')['queries'];qmap={q['query_id']:q for q in queries}
rubrics=P.F.read_json(BASE/'question_rubrics.json');rubric_by_query={q['query_id']:q for q in rubrics['records']}
assert len(qmap)==len(queries)==40
expected_ids={q+':S_ONLY:r1' for q in qmap}|{q+':'+profile+':'+mode+':r1' for q in qmap for profile in ['card_page_section_benefit','parent_child_bundle'] for mode in ['U','SEQ','PAR']}
for records in [rows,retrieval_rows,judgment_rows]:E.validate_identities([r['identity'] for r in records],expected_ids)
catalog={}
for name in ['anchor_source_judgments.json','verified_additional_cards.json']:
 for r in P.F.read_json(BASE/'source_review'/name)['records']:catalog[r['query_id'],r['card_key']]=r
source_questions={q for (q,k),v in catalog.items() if v['valid_for_original_question']}
assert len(source_questions)==37
inputs={str(p.relative_to(ROOT)):P.sha(p.read_bytes()) for p in [BASE/'execution_results.json',BASE/'question_source_judgments.json',BASE/'questions.json',BASE/'question_rubrics.json',BASE/'source_review/anchor_source_judgments.json',BASE/'source_review/verified_additional_cards.json',ROOT/'notebooks/data/54_new_question_confirmation/parser_results.json',ROOT/'notebooks/data/54_new_question_confirmation/source_gate_failure.json']}
inputs.update(upstream['source_input_sha256']);inputs.update(upstream['data_sha256'])
for p in ROOT.glob('data/rag/runtime/working/run_f2d29820c6dca642/documents/*/upstage/normalized.json'):
 inputs[str(p.relative_to(ROOT))]=P.sha(p.read_bytes())
for name,wanted in inputs.items():assert P.sha((ROOT/name).read_bytes())==wanted,name
inventory=dict(inputs=inputs,source_execution_sha256=P.sha(completed),logical_rows=280,questions=40,parser54_rows=96,split='posthoc_development_diagnosis',independent_human_review=False,new_API_requests=0)
X.frozen(PATH/'input_inventory.json',inventory)
print('원문·기존실험 입력 검증 완료:40질문/280결과,54추출96. 새API0.')

원문·기존실험 입력 검증 완료:40질문/280결과,54추출96. 새API0.


In [2]:
def join_by_card(evidence):
 result=collections.defaultdict(list)
 for e in evidence:result[e['card_key']].append(e['text'])
 return {k:'\n\n'.join(v) for k,v in result.items()}

def bound_case(row):
 retrieval=retrievals[row['identity']]
 assert old_judgments[row['identity']]['answer_sha256']==P.sha(row)
 assert row['retrieval_sha256']==P.sha(retrieval['result'])
 evidence=retrieval['result']['evidence'];input_evidence=[];proofs=[]
 if row['request_keys']:
  for key in row['request_keys']:
   raw_path=BASE/'api_cache'/(key+'.json');raw=P.F.read_json(raw_path)
   request=raw['request'];assert P.sha(request['input'])==row['input_sha256']
   payload=json.loads(request['input'][0]['content'])
   assert payload['standalone_query']==qmap[row['query_id']]['question']
   assert len(payload['evidence'])==len(evidence)
   restored=[]
   for index,(sent,saved) in enumerate(zip(payload['evidence'],evidence,strict=True),1):
    assert sent['evidence_id']=='e'+str(index)
    assert all(sent[k]==saved[k] for k in ['text','card_name','issuer'])
    restored.append({**sent,'card_key':saved['card_key']})
   input_evidence=restored
   proofs.append(dict(path=str(raw_path.relative_to(ROOT)),sha256=P.sha(raw_path.read_bytes())))
 else:assert not evidence
 answer=row.get('answer') or {}
 answer_cards=E.answer_by_card(answer,{e['card_key'] for e in evidence})
 return dict(identity=row['identity'],query_id=row['query_id'],config_id=row['profile']+':'+row['mode'],
  original_verdict=old_judgments[row['identity']]['question_verdict'],
  binding=dict(retrieval_sha256=P.sha(retrieval['result']),input_sha256=row['input_sha256'],answer_sha256=P.sha(row)),
  retrieved_text=join_by_card(evidence),input_text=join_by_card(input_evidence),
  answer_text=answer_cards,api_proofs=proofs)

cases={r['identity']:bound_case(r) for r in rows}
E.validate_identities(list(cases),expected_ids)
transport=[]
for identity,case in cases.items():
 if case['retrieved_text']!=case['input_text']:transport.append(identity)
assert not transport
row_inventory=[{k:c[k] for k in ['identity','query_id','config_id','original_verdict','binding','api_proofs']} for c in cases.values()]
X.frozen(PATH/'row_inventory.json',dict(records=row_inventory,transport_mismatch_ids=transport))
print('280개 실제 요청·입력·근거·답변 결속 확인. 전달 본문 차이0.')

def query_cases(query_id):
 return dict(question=qmap[query_id],rubric=rubric_by_query[query_id],cases=[c for c in cases.values() if c['query_id']==query_id])

280개 실제 요청·입력·근거·답변 결속 확인. 전달 본문 차이0.


## 원문 사실과 판정
원문 사실표 `source_facts.json`에 카드·대상·조건·기간·단위·예외 관계를 보존한 문장을 작성한다. 직접 요구/중요 조건/부가 정보와 원문 PDF 경로·SHA·페이지·발췌를 포함한다. `required_by_identity`는 각 조건의 필수사실 목록이며 판정 파일에서 임의 삭제할 수 없다.

`adjudications/<query_id>.json`은7개조건의 실제 검색·입력·답변SHA에 판정을 묶는다. 각 사실의 supported/absent/ambiguous/contradicted 상태·실제 인용·원인을 기록한다. 부가정보가 없다는 이유로 필수사실 실패를 만들지 않는다. source uncertain은별도 공개하며확정성공으로세지않는다.

전체280개 판정을 마치기 전에는 전체성적을출력하지 않는다. 명시적 실패가없어도카드근거의충분성이자동입증되는것은아니다.

In [4]:
reference_path=PATH/'source_facts.json'
judgment_paths=sorted((PATH/'adjudications').glob('*.json'))
if not reference_path.exists() or len(judgment_paths)!=40:
 print('원문 사실표·40질문 판정 대기. 전체 근거회수/반영 성적 미집계.')
 P.F.write_json(PATH/'completion_status.json',dict(status='awaiting_fact_source_adjudication',bound_rows=len(cases),expected=280,new_API_requests=0,operating_changes=False))
else:
 reference=P.F.read_json(reference_path);facts={f['fact_id']:f for f in reference['facts']}
 assert set(reference['required_by_identity'])==set(cases)
 for identity,required in reference['required_by_identity'].items():
  assert bool(required)==(cases[identity]['query_id'] in source_questions)
 annotations=[r for p in judgment_paths for r in P.F.read_json(p)['records']]
 assert len(annotations)==280 and len({a['identity'] for a in annotations})==280
 assert {a['identity'] for a in annotations}==set(cases)
 for fact in facts.values():
  for source in fact['source_support']:
   assert source['path']=='data/raw/'+fact['card_key']+'.pdf'
   if fact['source_status']=='verified':E.validate_source_support(source,ROOT)
 scored=[]
 for annotation in annotations:
  case=cases[annotation['identity']];expected=reference['required_by_identity'][annotation['identity']]
  assert all(annotation[k]==case[k] for k in ['query_id','config_id','original_verdict'])
  assert annotation['required_fact_ids']==expected
  assert annotation['reference_sha256']==P.sha(reference)
  expanded=[]
  for f in annotation['facts']:
   source=facts[f['fact_id']]
   expanded.append({**source,**f})
   assert f.get('card_key',source['card_key'])==source['card_key']
   assert all(f.get(k,source[k])==source[k] for k in ['source_support','source_status','statement'])
  row={**annotation,'facts':expanded}
  prose_spans=annotation.get('answer_prose_spans',[])
  if prose_spans:
   original=next(r for r in rows if r['identity']==annotation['identity'])
   evidence=retrievals[annotation['identity']]['result']['evidence']
   case={**case,'answer_text':E.answer_by_card(original.get('answer') or {},{e['card_key'] for e in evidence},prose_spans)}
  scored.append(E.score_row(row,retrieved_text=case['retrieved_text'],input_text=case['input_text'],answer_text=case['answer_text'],expected_binding=case['binding']))
 result=dict(records=scored,configurations=E.aggregate(scored),reference_sha256=P.sha(reference),input_inventory_sha256=P.sha(inventory),adjudication_sha256={str(p.relative_to(ROOT)):P.sha(p.read_bytes()) for p in judgment_paths},analysis_code_sha256={'notebooks/55_fact_retrieval_generation_diagnosis.ipynb':X.notebook_code_hash(ROOT/'notebooks/55_fact_retrieval_generation_diagnosis.ipynb'),'notebooks/scripts/55_fact_evaluation.py':P.sha((ROOT/'notebooks/scripts/55_fact_evaluation.py').read_bytes())},split='posthoc_development_diagnosis',independent_human_review=False)
 X.frozen(PATH/'evaluation_results.json',result)
 print(json.dumps(result['configurations'],ensure_ascii=False,indent=2))
 P.F.write_json(PATH/'completion_status.json',dict(status='primary_fact_diagnosis_completed_pending_material_audit',logical_rows=280,source_judgments=280,new_API_requests=0,operating_changes=False,production_adopted=False))

[
  {
    "config_id": "card_page_section_benefit:PAR",
    "question_denominator": 40,
    "logical_rows": 40,
    "macro": {
      "retrieval_recall": 0.9247747747747748,
      "input_recall": 0.9247747747747748,
      "answer_reflection": 0.972972972972973,
      "end_to_end_fulfillment": 0.913963963963964,
      "complete_retrieval": 0.7297297297297297,
      "complete_input": 0.7297297297297297
    },
    "macro_question_denominators": {
      "retrieval_recall": 37,
      "input_recall": 37,
      "answer_reflection": 37,
      "end_to_end_fulfillment": 37,
      "complete_retrieval": 37,
      "complete_input": 37
    },
    "diagnostic_micro_counts": {
      "source_reference_total": 128,
      "source_denominator": 125,
      "retrieved_supported": 114,
      "input_supported": 114,
      "answered_correctly": 112,
      "grounded_reflected": 112
    },
    "uncertain_source_fact_rows": 3
  },
  {
    "config_id": "card_page_section_benefit:SEQ",
    "question_denominator": 40

In [5]:
# 구조화0건과기술실패를구분한다. 아래집계는답변성능수치가아니다.
route52_path=BASE/'rerouted_repeat1_plans.json'
route52=P.F.read_json(route52_path)['records']
parser54_path=ROOT/'notebooks/data/54_new_question_confirmation/parser_results.json'
parser54=P.F.read_json(parser54_path)['records']
assert len(route52)==40 and len(parser54)==96
route_diagnostics={}
for label,records in [('52_development',route52),('54_preanswer',parser54)]:
 items=[]
 for record in records:
  parser=record['parser'];plan=parser['plan'];route=record['routing'];flags=[]
  if plan is not None:
   if not plan['groups']:flags.append('no_benefit_predicate')
   if plan['unsupported_filters']:flags.append('unsupported_explicit_filter')
   if any(r['cap_period']!='any' and r['min_cap_krw'] is None for group in plan['groups'] for r in group):flags.append('period_filter_without_numeric_cap')
  items.append(dict(query_id=parser['query_id'],repeat=parser['repeat'],parser_status=parser['status'],
   reason=route['reason'],db_attempted=route['db_attempted'],eligible_cards=route.get('eligible_cards',[]),
   diagnostic_flags=flags,plan=plan))
 route_diagnostics[label]=dict(logical_rows=len(items),question_count=len({r['query_id'] for r in items}),
  route_reason_counts=dict(collections.Counter(r['reason'] for r in items)),records=items)
route_diagnostics['source_bindings']={str(p.relative_to(ROOT)):P.sha(p.read_bytes()) for p in [route52_path,parser54_path]}
route_diagnostics['limitations']='원인후보표시이며완화조회·동의어보정·새검색·답변생성은하지않음. 결과0건과추출기술실패를구분. 원문정답유무와검색가능성은동일하지않음.'
X.frozen(PATH/'structured_route_diagnostics.json',route_diagnostics)
print(json.dumps({k:{f:v for f,v in value.items() if f!='records'} for k,value in route_diagnostics.items() if k in ['52_development','54_preanswer']},ensure_ascii=False,indent=2))


{
  "52_development": {
    "logical_rows": 40,
    "question_count": 40,
    "route_reason_counts": {
      "no_results": 15,
      "matched": 22,
      "ambiguous_or_unknown_product": 2,
      "parser_failure": 1
    }
  },
  "54_preanswer": {
    "logical_rows": 96,
    "question_count": 32,
    "route_reason_counts": {
      "no_results": 74,
      "matched": 21,
      "parser_failure": 1
    }
  }
}


In [6]:
audit_path=PATH/'source_review/material_recommendation_audit.json'
if not audit_path.exists():
 print('전체추가카드 중요조건감사 대기. 전체55완료로표시하지않음.')
else:
 audit=P.F.read_json(audit_path)
 policy=P.F.read_json(PATH/'source_review/denominator_policy.json')
 witness={r['identity']:set(r['witness_cards']) for r in policy['records']}
 expected_extra=set()
 for original in rows:
  answer=original.get('answer') or {}
  mentioned={item['card_key'] for item in [*answer.get('claims',[]),*answer.get('recommendations',[])]}
  expected_extra.update((original['identity'],key) for key in mentioned-witness[original['identity']])
 E.validate_material_audit(audit['records'],expected_extra,{k:c['binding'] for k,c in cases.items()})
 assert len(expected_extra)==89
 # Prior source validity and current answer coverage are separate checks.
 source_facts={f['fact_id']:f for f in P.F.read_json(PATH/'source_facts.json')['facts']}
 for record in audit['records']:
  case=cases[record['identity']]
  owning_text=case['answer_text'].get(record['card_key'],'')
  assert P.sha(owning_text)==record['answer_owning_card_text_sha256']
  assert record['query_id']==case['query_id'] and record['config_id']==case['config_id']
  for fid in record['source_fact_ids']:assert source_facts[fid]['card_key']==record['card_key']
  refs=record['source_references'];assert refs
  for ref in refs:
   if ref['kind']=='reused_52_source_card_review':
    allowed={str((BASE/'source_review'/name).relative_to(ROOT)) for name in ['anchor_source_judgments.json','verified_additional_cards.json']}
    assert ref['path'] in allowed
    original=[r for r in P.F.read_json(ROOT/ref['path'])['records'] if (r['query_id'],r['card_key'])==(record['query_id'],record['card_key'])]
    assert len(original)==1 and P.sha(original[0])==ref['record_sha256']
    assert original[0]['source_support']==ref['source_support']
    for support in ref['source_support']:
     assert support['path']=='data/raw/'+record['card_key']+'.pdf'
     assert P.sha((ROOT/support['path']).read_bytes())==support['sha256']
   elif ref['kind']=='direct_pdf_text_review':
    assert ref['path']=='data/raw/'+record['card_key']+'.pdf'
    E.validate_source_support(ref,ROOT)
   else:raise ValueError('Unknown material audit source proof')
  for finding in record['findings']:
   assert finding['reason'] and finding['source_reference_indices']
   assert all(type(i) is int and 0<=i<len(refs) for i in finding['source_reference_indices'])
   if finding['answer_quote']:assert E.canonical_text(finding['answer_quote']) in E.canonical_text(owning_text)
 primary=P.F.read_json(PATH/'evaluation_results.json')
 for name,sha in primary['adjudication_sha256'].items():assert P.sha((ROOT/name).read_bytes())==sha
 assert primary['reference_sha256']==P.sha(P.F.read_json(PATH/'source_facts.json'))
 subgroup={r['query_id']:r['subgroup'] for r in rubrics['records']}
 subgroup_results={label:E.aggregate([r for r in primary['records'] if subgroup[r['query_id']]==label]) for label in sorted(set(subgroup.values()))}
 issue_records=[]
 for r in primary['records']:
  if any(r[k] for k in ['retrieval_gaps','transport_gaps','generation_omissions','answer_contradictions','source_uncertain_fact_ids']):
   issue_records.append({k:r[k] for k in ['identity','query_id','config_id','retrieval_gaps','transport_gaps','generation_omissions','answer_contradictions','source_uncertain_fact_ids']})
 findings=collections.Counter(f['type'] for r in audit['records'] for f in r['findings'])
 analysis=dict(primary_evaluation_sha256=P.sha(primary),material_audit_sha256=P.sha(audit),
  subgroup_results=subgroup_results,issue_records=issue_records,
  material_review_status_counts=dict(collections.Counter(r['review_status'] for r in audit['records'])),
  material_finding_counts=dict(findings),extra_card_rows=89,independent_human_review=False,
  production_adopted=False,new_API_requests=0,split='posthoc_development_diagnosis')
 X.frozen(PATH/'analysis_summary.json',analysis)
 resource_path=PATH/'notebook_execution_resources.json'
 if not resource_path.exists():
  X.frozen(resource_path,dict(scope='one full offline notebook execution; source-adjudication and agent time excluded',
   wall_seconds=time.perf_counter()-_wall55,cpu_seconds=time.process_time()-_cpu55,
   peak_RSS_KiB=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
   new_API_requests=0,new_API_cost_usd=0,new_embeddings=0,
   TTFT='미측정',UI_latency='미측정',online_end_to_end='미측정'))
 P.F.write_json(PATH/'completion_status.json',dict(status='diagnosis_and_material_audit_completed_awaiting_final_review',
  logical_rows=280,expected=280,material_audit_rows=89,material_audit_completed=True,
  analysis_summary_sha256=P.sha(analysis),new_API_requests=0,operating_changes=False,production_adopted=False))
 print('주판정280행+추가카드89행 검토·결속확인. 최종보고서/독립검토대기.',dict(findings))


주판정280행+추가카드89행 검토·결속확인. 최종보고서/독립검토대기. {'material_condition_ambiguous': 5, 'material_condition_omitted': 27, 'nonmatching_alternative_disclosed': 2, 'ineligible_recommendation': 1, 'relation_scope_contradiction': 1, 'direct_requirement_unresolved': 1}
